# 03 — Default Credential Testing

| Field | Value |
|-------|-------|
| **MITRE ATT&CK ID** | T1078 — Valid Accounts |
| **Tactic** | Initial Access |
| **Difficulty** | Beginner |
| **Estimated Time** | 30 minutes |

## Threat Context: SolarWinds — The `solarwinds123` Password

In 2020, researchers discovered that a critical SolarWinds FTP server had been protected with the password `solarwinds123`. While the full SolarWinds supply-chain compromise (SUNBURST) involved a sophisticated backdoor inserted into Orion software updates, the weak default-style password on infrastructure systems illustrated a pervasive problem: **default and trivially guessable credentials remain one of the most common initial access vectors**.

Attackers routinely scan for services running with factory-default credentials — routers, IoT devices, admin panels, databases, and CI/CD systems. Shodan and Censys queries reveal thousands of internet-facing services still using `admin/admin`, `root/toor`, or vendor defaults.

## What You'll Understand After This

- How attackers build and use **default credential dictionaries** to test authentication endpoints
- Why **credential stuffing and password spraying** succeed against services with unchanged defaults
- How to **detect and defend** against default-credential attacks using logging, lockout policies, and credential audits

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Environment setup and imports

import http.server
import threading
import base64
import json
import time
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from collections import Counter

print("[+] Imports loaded successfully")
print("[i] This notebook demonstrates default credential testing against a LOCAL dummy HTTP service.")
print("[i] No real systems are targeted.")

### Step 1 — Build a Default Credential Dictionary

Attackers maintain lists of factory-default credentials for common services. These are publicly available (e.g., CIRT.net, SecLists). Here we create a small educational sample.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — sample default credential pairs
# These are well-known factory defaults, NOT real credentials

DEFAULT_CREDENTIALS = [
    {"username": "admin", "password": "admin", "service": "Generic Admin Panel"},
    {"username": "admin", "password": "password", "service": "Generic Admin Panel"},
    {"username": "admin", "password": "1234", "service": "Generic Router"},
    {"username": "root", "password": "root", "service": "Linux Default"},
    {"username": "root", "password": "toor", "service": "Kali Linux"},
    {"username": "admin", "password": "admin123", "service": "Web Application"},
    {"username": "user", "password": "user", "service": "Generic Service"},
    {"username": "test", "password": "test", "service": "QA Environment"},
    {"username": "guest", "password": "guest", "service": "Guest Access"},
    {"username": "admin", "password": "default", "service": "Network Device"},
]

print(f"[+] Loaded {len(DEFAULT_CREDENTIALS)} default credential pairs")
for cred in DEFAULT_CREDENTIALS[:3]:
    print(f"    {cred['username']}:{cred['password']} — {cred['service']}")

### Step 2 — Spin Up a Local Dummy HTTP Service

We create a simple HTTP server on `127.0.0.1` that accepts Basic Auth. It only accepts one credential pair (`admin`/`admin123`), simulating a service left with a default password.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — local dummy auth server

VALID_USER = "admin"
VALID_PASS = "admin123"  # Simulated default credential
SERVER_PORT = 18881

class DummyAuthHandler(http.server.BaseHTTPRequestHandler):
    """Simple HTTP handler that checks Basic Auth."""
    def do_GET(self):
        auth_header = self.headers.get("Authorization", "")
        if auth_header.startswith("Basic "):
            decoded = base64.b64decode(auth_header[6:]).decode("utf-8")
            username, password = decoded.split(":", 1)
            if username == VALID_USER and password == VALID_PASS:
                self.send_response(200)
                self.end_headers()
                self.wfile.write(b'{"status": "authenticated"}')
                return
        self.send_response(401)
        self.end_headers()
        self.wfile.write(b'{"status": "unauthorized"}')

    def log_message(self, format, *args):
        pass  # Suppress server logs in notebook

server = http.server.HTTPServer(("127.0.0.1", SERVER_PORT), DummyAuthHandler)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print(f"[+] Dummy auth server running on http://127.0.0.1:{SERVER_PORT}")

### Step 3 — Test Default Credentials Against the Service

This simulates what an attacker does: iterate through a credential list and record which pair grants access. In the real world, tools like Hydra, Medusa, or custom scripts automate this at scale.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — testing default credentials against LOCAL service

results = []

for cred in DEFAULT_CREDENTIALS:
    username = cred["username"]
    password = cred["password"]
    token = base64.b64encode(f"{username}:{password}".encode()).decode()
    req = Request(
        f"http://127.0.0.1:{SERVER_PORT}/",
        headers={"Authorization": f"Basic {token}"}
    )
    try:
        response = urlopen(req)
        status = response.getcode()
    except HTTPError as e:
        status = e.code

    result = {
        "username": username,
        "password": password,
        "status": status,
        "success": status == 200
    }
    results.append(result)
    marker = "SUCCESS" if result["success"] else "FAILED"
    print(f"  [{marker}] {username}:{password} -> HTTP {status}")

successful = [r for r in results if r["success"]]
print(f"\n[+] Scan complete: {len(successful)}/{len(results)} credentials valid")

# Shut down the dummy server
server.shutdown()

### Step 4 — Analyze Common Default Passwords

Beyond our small list, let's look at password frequency patterns from known default credential databases. This illustrates why certain passwords appear over and over.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — common default password frequency analysis

# Extended list representing frequency in real default-credential databases
common_default_passwords = [
    "admin", "admin", "admin", "admin", "admin",
    "password", "password", "password", "password",
    "1234", "1234", "1234",
    "12345", "12345",
    "default", "default",
    "root", "root",
    "changeme", "changeme",
    "pass", "pass",
    "test",
    "guest",
    "admin123",
    "toor",
    "letmein",
    "welcome",
]

password_counts = Counter(common_default_passwords)
print("[+] Most common default passwords (by frequency in vendor databases):")
for pwd, count in password_counts.most_common():
    bar = '█' * count
    print(f"  {pwd:<12} {bar} ({count})")

### Visualization — Most Common Default Passwords

In [ ]:
import matplotlib.pyplot as plt

# Bar chart of most common default passwords
passwords_sorted = password_counts.most_common()
labels = [p[0] for p in passwords_sorted]
counts = [p[1] for p in passwords_sorted]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels[::-1], counts[::-1], color="#e74c3c", edgecolor="#c0392b")
ax.set_xlabel("Frequency in Default Credential Databases")
ax.set_title("Most Common Default Passwords Across Vendor Devices")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(count), va="center", fontsize=9)

plt.tight_layout()
plt.show()
print("[+] Visualization complete")

## Defender's Perspective — Indicators of Compromise

| Indicator | Type | Detection Method |
|-----------|------|------------------|
| Multiple failed login attempts from single IP | Behavioral | Auth log monitoring, SIEM correlation |
| Known default usernames in auth logs (`admin`, `root`, `guest`) | Account-based | Username watchlist alerting |
| Rapid sequential login attempts across accounts | Behavioral | Rate limiting, account lockout policies |
| Successful auth with a known default password | Credential | Credential audit tools (e.g., Nessus default cred checks) |
| Access from unexpected geographic locations after default cred use | Network | GeoIP correlation on auth events |

## Article Seed

**Title:** *The Invisible Front Door: How Default Credentials Still Breach Enterprise Networks*

**Opening Paragraph:**
In an era of advanced persistent threats and zero-day exploits, it may seem surprising that one of the most effective initial access techniques remains embarrassingly simple: trying the default password. From IoT devices to enterprise admin panels, factory credentials continue to provide attackers with a silent entry point that bypasses every firewall and IDS in the network.

**Section Headers:**
1. The Scale of the Problem: Default Credentials in the Wild
2. Anatomy of a Default Credential Attack
3. Case Studies: SolarWinds, Mirai Botnet, and Beyond
4. Building a Default Credential Defense Program

**Medium Tags:** `cybersecurity`, `initial-access`, `credential-security`, `mitre-attack`, `penetration-testing`

In [ ]:
# Self-check assertions
# Verify the notebook's key concepts were demonstrated

# 1. We built a credential dictionary
assert len(DEFAULT_CREDENTIALS) >= 5, "Credential dictionary should have at least 5 entries"

# 2. We tested credentials and got results
assert len(results) == len(DEFAULT_CREDENTIALS), "Should have tested all credentials"

# 3. Exactly one credential pair should have succeeded (admin/admin123)
assert len(successful) == 1, f"Expected 1 successful login, got {len(successful)}"
assert successful[0]["username"] == "admin" and successful[0]["password"] == "admin123"

print("[+] All self-check assertions passed!")